# Text Evaluation Module 
## Social Media Strategy Evaluator 

In [16]:
# ==============================
# 1. INSTALL & IMPORTS
# ==============================
!pip install transformers datasets scikit-learn --quiet

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import DistilBertTokenizer, DistilBertModel
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [17]:
# ==============================
# 1. IMPORTS
# ==============================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

from xgboost import XGBClassifier

In [18]:
# ==============================
# 2. LOAD DATASET
# ==============================

from pathlib import Path
import kagglehub
from kagglehub import KaggleDatasetAdapter

slug = 'kundanbedmutha/instagram-analytics-dataset'
preferred = ['Instagram_Analytics.csv', 'instagram_analytics.csv', 'instagram_dataset.csv', 'instagram.csv']

df = None
for file_name in preferred:
    try:
        df_try = kagglehub.load_dataset(
            KaggleDatasetAdapter.PANDAS,
            slug,
            file_name
        )
        if isinstance(df_try, pd.DataFrame) and not df_try.empty:
            df = df_try.copy()
            print(f'Loaded via adapter: {file_name}')
            break
    except Exception:
        pass

if df is None:
    dataset_dir = Path(kagglehub.dataset_download(slug))
    csv_files = sorted(dataset_dir.rglob('*.csv'))
    if not csv_files:
        raise RuntimeError(f'No CSV files found in {dataset_dir}')

    preferred_lower = [p.lower() for p in preferred]
    chosen = None
    for f in csv_files:
        if f.name.lower() in preferred_lower:
            chosen = f
            break
    if chosen is None:
        chosen = max(csv_files, key=lambda p: p.stat().st_size)

    df = pd.read_csv(chosen)
    print(f'Loaded via pd.read_csv: {chosen.name}')

df.columns = [str(c).strip() for c in df.columns]
raw_df = df.copy()  # Keep an untouched copy for repeatable reruns
print('Dataset shape:', df.shape)
df.head()

# ==============================
# 3. CHECK COLUMNS
# ==============================

print(df.columns)

Loaded from cache: C:\Users\MSI\.cache\kagglehub\datasets\kundanbedmutha\instagram-analytics-dataset\versions\3\Instagram_Analytics.csv
Dataset shape: (29999, 23)
Index(['post_id', 'account_id', 'account_type', 'follower_count', 'media_type',
       'content_category', 'traffic_source', 'has_call_to_action',
       'post_datetime', 'post_date', 'post_hour', 'day_of_week', 'likes',
       'comments', 'shares', 'saves', 'reach', 'impressions',
       'engagement_rate', 'followers_gained', 'caption_length',
       'hashtags_count', 'performance_bucket_label'],
      dtype='str')


In [19]:
# ==============================
# 4. CLEAN DATA + FEATURE ENGINEERING (USER-REQUESTED INPUTS ONLY)
# ==============================

base_df = raw_df.copy() if 'raw_df' in globals() else df.copy()

# Keep only features requested by user + target
requested_cols = [
    'content_category',      # content type
    'account_type',          # account type
    'follower_count',
    'post_date',             # date feature
    'post_hour',             # hour feature
    'has_call_to_action',
    'comments',
    'shares',
    'saves',
    'performance_bucket_label',
]

# Some datasets may miss post_date but have post_datetime
if 'post_date' not in base_df.columns and 'post_datetime' in base_df.columns:
    base_df['post_date'] = base_df['post_datetime']

available = [c for c in requested_cols if c in base_df.columns]
df = base_df[available].dropna().copy()

# Numeric cleanup
for col in ['follower_count', 'post_hour', 'comments', 'shares', 'saves']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna().copy()

# CTA to 0/1
if 'has_call_to_action' in df.columns:
    df['has_call_to_action'] = df['has_call_to_action'].astype(str).str.lower().map({
        'true': 1, '1': 1, 'yes': 1,
        'false': 0, '0': 0, 'no': 0
    }).fillna(0).astype(int)

# Time features from post_date
if 'post_date' in df.columns:
    dt = pd.to_datetime(df['post_date'], errors='coerce')
    df['day_of_week'] = dt.dt.day_name()
    df['month'] = dt.dt.month.fillna(0).astype(int)
    df['is_weekend'] = dt.dt.dayofweek.isin([5, 6]).fillna(False).astype(int)
    df = df.dropna(subset=['day_of_week'])

# Hour as cyclical feature
df['post_hour'] = df['post_hour'].clip(0, 23)
df['hour_sin'] = np.sin(2 * np.pi * df['post_hour'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['post_hour'] / 24.0)

# Keep this as a mild scale transform only
df['log_follower_count'] = np.log1p(df['follower_count'])

# post_date is used only to derive temporal features, then removed
if 'post_date' in df.columns:
    df = df.drop(columns=['post_date'])

# ==============================
# 5. TARGET PREPARATION
# ==============================

target_col = 'performance_bucket_label' if 'performance_bucket_label' in df.columns else 'label'
df = df.rename(columns={target_col: 'label'})

# Collapse to fewer classes for a more reliable target
binary_map = {
    'low': 'low',
    'medium': 'low',
    'high': 'high',
    'viral': 'high',
}
df['label'] = df['label'].map(binary_map)
df = df.dropna(subset=['label']).copy()

le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])
print('Classes:', le.classes_)
print('Class distribution:')
print(df['label'].value_counts(normalize=True).sort_index())

# ==============================
# 6. SPLIT FEATURES / TARGET
# ==============================

X = df.drop('label', axis=1)
y = df['label']

# ==============================
# 7. ENCODING
# ==============================

categorical_cols = [
    c for c in ['content_category', 'account_type', 'day_of_week']
    if c in X.columns
]
X = pd.get_dummies(X, columns=categorical_cols, drop_first=False)

# Save schema for inference
MODEL_INPUT_COLUMNS = list(X.columns)

print('Final feature shape:', X.shape)
print('engagement_rate in features:', 'engagement_rate' in X.columns)
print('Using categorical columns:', categorical_cols)

Classes: ['high' 'low']
Class distribution:
label
0    0.500017
1    0.499983
Name: proportion, dtype: float64
Final feature shape: (29999, 30)
engagement_rate in features: False
Using categorical columns: ['content_category', 'account_type', 'day_of_week']


In [20]:
# ==============================
# 8. TRAIN / TEST SPLIT
# ==============================

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ==============================
# 9. MODEL TRAINING (STRICT FEATURES, BINARY TARGET)
# ==============================

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score

candidates = []

# XGBoost binary classifiers
xgb_a = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    n_estimators=900,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=1,
    reg_alpha=0.05,
    reg_lambda=1.5,
)
xgb_a.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
xgb_a_acc = accuracy_score(y_val, xgb_a.predict(X_val))
print(f'XGBoost A accuracy: {xgb_a_acc:.4f}')
candidates.append(('XGBoost A', xgb_a_acc, xgb_a))

xgb_b = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    n_estimators=1200,
    max_depth=4,
    learning_rate=0.02,
    subsample=1.0,
    colsample_bytree=0.8,
    min_child_weight=1,
    reg_alpha=0.0,
    reg_lambda=2.0,
)
xgb_b.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
xgb_b_acc = accuracy_score(y_val, xgb_b.predict(X_val))
print(f'XGBoost B accuracy: {xgb_b_acc:.4f}')
candidates.append(('XGBoost B', xgb_b_acc, xgb_b))

# Tree bagging variants
rf = RandomForestClassifier(
    n_estimators=900,
    max_depth=None,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
rf_acc = accuracy_score(y_val, rf.predict(X_val))
print(f'RandomForest accuracy: {rf_acc:.4f}')
candidates.append(('RandomForest', rf_acc, rf))

et = ExtraTreesClassifier(
    n_estimators=1200,
    max_depth=None,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
)
et.fit(X_train, y_train)
et_acc = accuracy_score(y_val, et.predict(X_val))
print(f'ExtraTrees accuracy: {et_acc:.4f}')
candidates.append(('ExtraTrees', et_acc, et))

# Linear baseline
lr = LogisticRegression(
    max_iter=3000,
    class_weight='balanced',
)
lr.fit(X_train, y_train)
lr_acc = accuracy_score(y_val, lr.predict(X_val))
print(f'LogisticRegression accuracy: {lr_acc:.4f}')
candidates.append(('LogisticRegression', lr_acc, lr))

# Fast histogram boosting
hgb = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_depth=6,
    max_iter=300,
    min_samples_leaf=20,
    random_state=42,
)
hgb.fit(X_train, y_train)
hgb_acc = accuracy_score(y_val, hgb.predict(X_val))
print(f'HistGradientBoosting accuracy: {hgb_acc:.4f}')
candidates.append(('HistGradientBoosting', hgb_acc, hgb))

best_name, best_acc, best_model = max(candidates, key=lambda t: t[1])

# Prefer an explainable tree model if it is very close to the best score.
explainable_candidates = [
    (name, acc, mdl) for name, acc, mdl in candidates
    if hasattr(mdl, 'feature_importances_')
 ]
explainable_candidates = sorted(explainable_candidates, key=lambda t: t[1], reverse=True)
explainable_name, explainable_acc, explainable_model = explainable_candidates[0]

if best_acc - explainable_acc <= 0.01:
    model = explainable_model
    selected_name = explainable_name
    selected_acc = explainable_acc
else:
    model = best_model
    selected_name = best_name
    selected_acc = best_acc

# Overfitting check
train_acc = accuracy_score(y_train, model.predict(X_train))
gap = train_acc - selected_acc

print('Best predictive model:', best_name)
print('Best predictive accuracy:', round(best_acc, 4))
print('Selected model for reporting:', selected_name)
print('Selected validation accuracy:', round(selected_acc, 4))
print('Train accuracy:', round(train_acc, 4))
print('Generalization gap (train - val):', round(gap, 4))
if gap > 0.12:
    print('Warning: possible overfitting (large train/val gap).')

XGBoost A accuracy: 0.7013
XGBoost B accuracy: 0.7052
RandomForest accuracy: 0.7012
ExtraTrees accuracy: 0.6888


XGBoost A accuracy: 0.7013
XGBoost B accuracy: 0.7052
RandomForest accuracy: 0.7012
ExtraTrees accuracy: 0.6888


c:\Users\MSI\Documents\3ia\AI project  consulting\notebooks marketing\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression accuracy: 0.6972
HistGradientBoosting accuracy: 0.7085
Best predictive model: HistGradientBoosting
Best predictive accuracy: 0.7085
Selected model for reporting: XGBoost B
Selected validation accuracy: 0.7052
Train accuracy: 0.7252
Generalization gap (train - val): 0.02


In [21]:
# ==============================
# 10. EVALUATION
# ==============================

preds = model.predict(X_val)

accuracy = accuracy_score(y_val, preds)

print('Accuracy:', accuracy)
print('\nClassification Report:\n')
print(classification_report(y_val, preds))

# ==============================
# 11. FEATURE IMPORTANCE
# ==============================

#import matplotlib.pyplot as plt

#if hasattr(model, 'feature_importances_'):
#    importance = model.feature_importances_
#    features = X.columns

#    indices = np.argsort(importance)[-10:]

#    plt.figure(figsize=(8, 6))
#    plt.barh(range(len(indices)), importance[indices])
#    plt.yticks(range(len(indices)), [features[i] for i in indices])
#    plt.title('Top Features')
#    plt.show()
#else:
#    print('Selected model does not expose feature_importances_.')
#    print('Use one of the tree models above if you need a feature-importance chart.')

Accuracy: 0.7051666666666667

Classification Report:

              precision    recall  f1-score   support

           0       0.70      0.73      0.71      3000
           1       0.71      0.68      0.70      3000

    accuracy                           0.71      6000
   macro avg       0.71      0.71      0.71      6000
weighted avg       0.71      0.71      0.71      6000



In [22]:
# ==============================
# 12. PREDICTION FUNCTION
# ==============================

def _prepare_single_input(input_dict):
    """Prepare one user input row with the same transforms used in training."""
    inp = pd.DataFrame([input_dict]).copy()

    # Ensure required fields exist
    defaults = {
        'follower_count': 0,
        'post_hour': 12,
        'comments': 0,
        'shares': 0,
        'saves': 0,
        'has_call_to_action': 0,
        'content_category': 'Unknown',
        'account_type': 'unknown',
        'post_date': '2024-01-01',
    }
    for col, default in defaults.items():
        if col not in inp.columns:
            inp[col] = default

    # Numeric casting
    for col in ['follower_count', 'post_hour', 'comments', 'shares', 'saves']:
        inp[col] = pd.to_numeric(inp[col], errors='coerce').fillna(0)

    # CTA to 0/1
    inp['has_call_to_action'] = inp['has_call_to_action'].astype(str).str.lower().map({
        'true': 1, '1': 1, 'yes': 1,
        'false': 0, '0': 0, 'no': 0
    }).fillna(0).astype(int)

    # Time features
    dt = pd.to_datetime(inp['post_date'], errors='coerce')
    inp['day_of_week'] = dt.dt.day_name().fillna('Unknown')
    inp['month'] = dt.dt.month.fillna(0).astype(int)
    inp['is_weekend'] = dt.dt.dayofweek.isin([5, 6]).fillna(False).astype(int)

    # Hour as cyclical feature
    inp['post_hour'] = inp['post_hour'].clip(0, 23)
    inp['hour_sin'] = np.sin(2 * np.pi * inp['post_hour'] / 24.0)
    inp['hour_cos'] = np.cos(2 * np.pi * inp['post_hour'] / 24.0)

    # Keep only training-like columns before encoding
    keep_cols = [
        'content_category', 'account_type', 'follower_count', 'post_hour',
        'has_call_to_action', 'comments', 'shares', 'saves',
        'day_of_week', 'month', 'is_weekend', 'hour_sin', 'hour_cos',
        'log_follower_count',
    ]
    inp['log_follower_count'] = np.log1p(inp['follower_count'])
    inp = inp[keep_cols]

    # Apply same encoding and align columns
    inp = pd.get_dummies(inp, columns=['content_category', 'account_type', 'day_of_week'], drop_first=False)
    inp = inp.reindex(columns=MODEL_INPUT_COLUMNS, fill_value=0)
    return inp

def predict_post_performance(input_dict):
    input_df = _prepare_single_input(input_dict)
    pred = model.predict(input_df)
    return le.inverse_transform(pred)[0]

In [23]:
# ==============================
# 13. TEST
# ==============================

sample_input = {
    'follower_count': 10000,
    'content_category': 'Technology',
    'account_type': 'creator',
    'post_date': '2024-05-13',
    'post_hour': 18,
    'has_call_to_action': 1,
    'comments': 120,
    'shares': 75,
    'saves': 190,
}

print('Predicted performance:', predict_post_performance(sample_input))

Predicted performance: high


In [ ]:
# CELL XX — Screenshot → full pipeline demo
# Update the screenshot path below to a real file on your machine
from engagement_pipeline import predict_engagement_from_screenshot
import json
screenshot_path = input('Enter the full path to your screenshot: ').strip()
outputs_dir = r'c:\\Users\\MSI\\Documents\\3ia\\AI project  consulting\\notebooks marketing\\outputs'
if not screenshot_path:
    raise ValueError('Please enter a valid screenshot path.')
res = predict_engagement_from_screenshot(screenshot_path, outputs_dir=outputs_dir)
print('Pipeline result (JSON):')
print(json.dumps(res, indent=2))